# 4.4 LLM multimodal en zero-shot : Datasets 1 et 2

On envoie l'image à un LLM local (`gemma4`, via Ollama) avec la liste des classes, sans entraînement.
Chaque appel prend plusieurs secondes, donc 10 images par classe. Les réponses sont enregistrées et
relues ici ; sans ce fichier, le notebook interroge Ollama.

In [1]:
import os
import random

import ollama
import pandas as pd

from outils_4 import ARTEFACTS_30K, ARTEFACTS_DATASET_1


def classify_with_llm(chemin, classes):
    prompt = (f"Voici un document scanné. Parmi ces catégories : {', '.join(classes)}, "
              "laquelle correspond le mieux ? Réponds uniquement par le nom exact de la catégorie.")
    reponse = ollama.chat(model="gemma4:latest", messages=[{"role": "user", "content": prompt, "images": [chemin]}])
    texte = reponse["message"]["content"].strip().lower()
    return next((classe for classe in classes if classe in texte), "inconnu")


def evaluate_llm(dossier, classes, fichier, par_classe=10):
    """réponses du LLM sur quelques images par classe (relues dans le cache si déjà posées)"""
    if os.path.exists(fichier):
        return pd.read_csv(fichier)
    random.seed(1)
    echantillon = [(f"{dossier}/{classe}/{nom}", classe) for classe in classes
                   for nom in random.sample(os.listdir(f"../{dossier}/{classe}"), par_classe)]
    random.shuffle(echantillon)
    resultats = pd.DataFrame([{"filepath": f, "label": classe, "prediction_llm": classify_with_llm("../" + f, classes)}
                              for f, classe in echantillon])
    resultats.to_csv(fichier, index=False)
    return resultats

## 1. Dataset 1 (3 classes, 30 images)

In [2]:
d1 = evaluate_llm("data/document-classification-dataset_1", ["email", "resume", "scientific_publication"],
                  f"{ARTEFACTS_DATASET_1}/llm_zero_shot.csv")
print(f"Accuracy : {(d1['label'] == d1['prediction_llm']).mean():.1%} ({len(d1)} images)")
pd.crosstab(d1["label"], d1["prediction_llm"], rownames=["réel"], colnames=["prédit"])

Accuracy : 93.3% (30 images)


prédit,email,resume,scientific_publication
réel,,,
email,10,0,0
resume,0,8,2
scientific_publication,0,0,10


## 2. Dataset 2, 1er extrait (6 classes, 60 images)

In [3]:
d2 = evaluate_llm("data/rvl_cdip_filtered_dataset_2",
                  ["email", "form", "invoice", "presentation", "resume", "scientific_report"],
                  f"{ARTEFACTS_30K}/llm_zero_shot.csv")
print(f"Accuracy : {(d2['label'] == d2['prediction_llm']).mean():.1%} ({len(d2)} images)")
pd.crosstab(d2["label"], d2["prediction_llm"], rownames=["réel"], colnames=["prédit"])

Accuracy : 73.3% (60 images)


prédit,email,form,invoice,presentation,resume,scientific_report
réel,,,,,,
email,9,1,0,0,0,0
form,1,8,0,0,0,1
invoice,0,5,5,0,0,0
presentation,0,1,0,2,0,7
resume,0,0,0,0,10,0
scientific_report,0,0,0,0,0,10


Sur 6 classes, il fait les mêmes erreurs que nos autres modèles : présentations prises pour des
rapports scientifiques, factures pour des formulaires.